# Publicar libro (GitHub Pages) y dashboard (Render)
1. Sube `proyecto_energia.zip`.
2. Crea en GitHub un repositorio vacío (p. ej. `ML_energia_caribe`).
3. Corre las celdas. El token se pide con `getpass` y nunca se imprime.

In [ ]:
!pip -q install jupyter-book==1.0.3 ghp-import
import glob, os, zipfile
z = max(glob.glob("/content/proyecto_energia*.zip"), key=os.path.getmtime)  # toma el zip más reciente, aunque Colab lo renombre
zipfile.ZipFile(z).extractall("/content/repo"); os.chdir("/content/repo"); print(z, os.listdir())

In [ ]:
import getpass, base64, subprocess
USUARIO = "jserrano3010"          # cambia si es otro
REPO = "ML_energia_caribe"           # nombre del repositorio vacío
tok = getpass.getpass("Token de GitHub (no se muestra): ")
os.environ["GIT_CONFIG_COUNT"] = "1"
os.environ["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
os.environ["GIT_CONFIG_VALUE_0"] = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{tok}".encode()).decode()
del tok
def git(*a):
    r = subprocess.run(["git", *a], capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(os.environ["GIT_CONFIG_VALUE_0"], "***")[-800:])
git("init", "-b", "main"); git("config", "user.email", "jairo.andres.serrano@gmail.com"); git("config", "user.name", "Jairo Serrano")
git("add", "."); git("commit", "-m", "Proyecto final ML: demanda eléctrica Caribe")
git("remote", "add", "origin", f"https://github.com/{USUARIO}/{REPO}.git"); git("push", "-u", "origin", "main", "--force")

In [ ]:
!jupyter-book build libro
!ghp-import -n -p -f libro/_build/html
print(f"Libro: https://{USUARIO}.github.io/{REPO}/  (en GitHub: Settings → Pages → Deploy from a branch → gh-pages / root)")

**Render:** *New → Blueprint* → conecta el repositorio `ML_energia_caribe` (toma `render.yaml`: build `pip install -r requirements.txt`, start `gunicorn app:server`, Python 3.13.16). O *New → Web Service* con esos mismos comandos. Cada push redespliega solo. La URL queda como `https://ml-energia-caribe.onrender.com` si el nombre está libre; si Render asigna otra, cámbiala en `libro/intro.md` y `libro/dashboard.md`.

## Paso final: poner la URL real de Render en el libro
1. En Render, cuando el servicio diga **Live**, copia la URL que aparece arriba a la izquierda (algo como `https://ml-energia-caribe.onrender.com` o `https://ml-energia-caribe-xxxx.onrender.com`).
2. Pégala en `URL_RENDER` en la celda de abajo y córrela. La celda reemplaza la URL en el libro, el README y el dashboard, recompila el libro y sube todo (GitHub Pages y Render se actualizan solos).

In [ ]:
URL_RENDER = "https://ml-energia-caribe.onrender.com"   # <- pega aquí la URL que te dio Render
VIEJA = "https://ml-energia-caribe.onrender.com"
import pathlib
for f in ["libro/intro.md", "libro/dashboard.md", "README.md"]:
    p = pathlib.Path(f); p.write_text(p.read_text(encoding="utf-8").replace(VIEJA, URL_RENDER), encoding="utf-8")
git("add", "."); git("commit", "-m", "URL de Render"); git("push", "origin", "main")
!jupyter-book build libro
!ghp-import -n -p -f libro/_build/html
print("Listo. Libro:", f"https://{USUARIO}.github.io/{REPO}/", "| Dashboard:", URL_RENDER)